# 01 — Feature Engineering

Run this notebook **first**. It loads the raw competition files, builds the engineered feature set, and
saves `train_features.csv` / `test_features.csv` — every other notebook in this set loads those two files
rather than redoing feature engineering, so all models train on exactly the same columns and fold splits.

`daily_screen_time_hours`, `social_media_hours`, and `weekend_screen_time` are by far the strongest
individual predictors (there's a sharp sigmoid-shaped jump in addiction rate around 6–9.5 daily screen-time
hours). `gender`, `stress_level`, and `academic_work_impact` carry essentially no signal on their own —
the screen-time/label relationship is identical across every group of each — but boosted trees can still
pick up minor conditional splits on them, so they're kept in as low-cost categorical features.

**Set `DATA_DIR` below** to the folder containing `train.csv`, `test.csv`, `sample_submission.csv`.

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "."   # <-- folder with train.csv / test.csv
OUT_DIR = "."     # <-- where to save train_features.csv / test_features.csv

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)
train.head()


(691369, 14) (296302, 13)


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [2]:
def engineer(df):
    df = df.copy()
    df['notif_per_hour'] = df['notifications_per_day'] / df['daily_screen_time_hours']
    df['app_per_hour'] = df['app_opens_per_day'] / df['daily_screen_time_hours']
    df['mins_per_appopen'] = (df['daily_screen_time_hours'] * 60) / df['app_opens_per_day']
    df['mins_per_notif'] = (df['daily_screen_time_hours'] * 60) / df['notifications_per_day']
    df['productivity'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 0.01)
    df['sleep_screen_sum'] = df['sleep_hours'] + df['daily_screen_time_hours']
    df['nonscreen_hours'] = 24 - df['daily_screen_time_hours'] - df['sleep_hours']
    df['screen_plus_weekend'] = df['daily_screen_time_hours'] + df['weekend_screen_time']
    df['screen_sleep_ratio'] = df['daily_screen_time_hours'] / df['sleep_hours']
    df['sm_ratio'] = df['social_media_hours'] / df['daily_screen_time_hours']
    df['game_ratio'] = df['gaming_hours'] / df['daily_screen_time_hours']
    df['work_ratio'] = df['work_study_hours'] / df['daily_screen_time_hours']
    df['screen_minus_work'] = df['daily_screen_time_hours'] - df['work_study_hours']
    df['weekday_weekend_ratio'] = df['daily_screen_time_hours'] / (df['weekend_screen_time'] + 0.01)
    df['screen_x_sm'] = df['daily_screen_time_hours'] * df['social_media_hours']
    df['screen_x_weekend'] = df['daily_screen_time_hours'] * df['weekend_screen_time']
    df['sm_x_weekend'] = df['social_media_hours'] * df['weekend_screen_time']
    df['screen_x_sleep'] = df['daily_screen_time_hours'] * df['sleep_hours']
    return df

train_fe = engineer(train)
test_fe = engineer(test)

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
            'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
            'notif_per_hour', 'app_per_hour', 'mins_per_appopen', 'mins_per_notif', 'productivity',
            'sleep_screen_sum', 'nonscreen_hours', 'screen_plus_weekend', 'screen_sleep_ratio',
            'sm_ratio', 'game_ratio', 'work_ratio', 'screen_minus_work', 'weekday_weekend_ratio',
            'screen_x_sm', 'screen_x_weekend', 'sm_x_weekend', 'screen_x_sleep']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
feat_cols = num_cols + cat_cols

# clean up any inf from the ratio features (near-zero denominators)
train_fe[num_cols] = train_fe[num_cols].replace([np.inf, -np.inf], np.nan)
test_fe[num_cols] = test_fe[num_cols].replace([np.inf, -np.inf], np.nan)

print("engineered feature count:", len(feat_cols))


engineered feature count: 30


In [3]:
save_cols_train = ['id'] + feat_cols + ['addicted_label']
save_cols_test = ['id'] + feat_cols

train_fe[save_cols_train].to_csv(f"{OUT_DIR}/train_features.csv", index=False)
test_fe[save_cols_test].to_csv(f"{OUT_DIR}/test_features.csv", index=False)
print("saved train_features.csv and test_features.csv to", OUT_DIR)


saved train_features.csv and test_features.csv to .
